### SmallTalk 1: Data, RAG and Baselines
**Can Agentic RAG and LoRA-Based DPO Help a 1.7B Model Close the Gap with Larger Models in Meeting Summarization?**

This notebook loads QMSum, builds the retriever, measures retrieval quality against the gold spans, and runs the baselines for every model: zero-shot, standard RAG and agentic RAG. The small model is Qwen3-1.7B, the bigger models are Qwen3-4B (same family) and Ministral 3 3B (another family), and Qwen3-8B is an extra reference.

### Setup
On Colab, choose **Runtime > Change runtime type > T4 GPU** first. Upload `smalltalk.py`, `config.py` and `requirements.txt` to `MyDrive/smalltalk/` (or keep them next to this notebook in VS Code). This cell also installs the libraries in `requirements.txt`.

In [1]:
import os, sys
os.environ['HF_HUB_DISABLE_XET'] = '1'   # the Xet downloader can hang on Colab, must be set before any Hugging Face import

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/smalltalk'  # results are saved on Drive so a disconnect does not lose them
else:
    WORK = os.getcwd()
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)   # so the notebook can import smalltalk.py and config.py
print('Working folder:', WORK)
for f in ['smalltalk.py', 'config.py', 'requirements.txt']:
    assert os.path.exists(f), f'Put {f} in the working folder first'

%pip install -q -r requirements.txt

Working folder: <project folder>


Note: you may need to restart the kernel to use updated packages.


### Config
All settings (models, hyperparameters, prompts) are in `config.py`. Edit that file, save it, and run this cell again. To change a setting for this notebook only, set it in a cell after this one.

`RUN_ON` sets where the notebook runs: `'local'` (GPU under 12 GB), `'colab'` (T4), `'server'` (remote GPU server) or `'auto'` to detect it. It only changes the batch size and which model sizes run, so the results stay comparable.

`DOWNLOADS` is the one folder that holds everything downloaded: the dataset in `QMSum/` and the base models in `models/`. What is already there is reused, and only what is missing is downloaded.

In [2]:
import json, glob, importlib
import pandas as pd
import config, smalltalk as st
importlib.reload(config); importlib.reload(st)   # picks up edits to config.py without a kernel restart
from config import *                              # every setting lives in config.py

st.set_seed(SEED)
os.makedirs('runs', exist_ok=True); os.makedirs('results', exist_ok=True)

PROFILE = st.run_profile(RUN_ON)
BATCH = PROFILE['batch_size']
# reuses the dataset and models that are already in the folder, downloads what is missing
st.setup_downloads(DOWNLOADS or PROFILE['downloads'])
# only needed if a model asks you to accept its terms on Hugging Face (keep it after setup_downloads)
# from huggingface_hub import login; login()

train_meetings, val_meetings, test_meetings = (st.load_split(s) for s in ['train', 'val', 'test'])
meetings_by_id = {m['id']: m for m in train_meetings + val_meetings + test_meetings}
test_examples = st.test_subset(st.build_examples(test_meetings), N_TEST)
print('Test questions used:', len(test_examples), '| K =', K, '| 4-bit:', LOAD_4BIT)

Running on: local (6.4 GB GPU), batch size 2, models up to 4.5B, training: True
Downloads folder: ~/smalltalk_downloads (928 GB free)
Models already downloaded: BAAI/bge-small-en-v1.5, Qwen/Qwen3-1.7B, mistralai/Ministral-3-3B-Instruct-2512-BF16


Test questions used: 150 | K = 8 | 4-bit: True


### 1. Data
QMSum has 232 meetings (product design, academic and parliament committee meetings). Each meeting has general questions ("Summarize the whole meeting") and specific questions with gold spans that mark which speaker turns hold the answer.

In [3]:
for name, ms in [('train', train_meetings), ('val', val_meetings), ('test', test_meetings)]:
    ex = st.build_examples(ms)
    words = [sum(len(t['text'].split()) for t in m['turns']) for m in ms]
    print(f"{name}: {len(ms)} meetings, {len(ex)} questions "
          f"({sum(e['qtype'] == 'general' for e in ex)} general), median meeting length {int(pd.Series(words).median())} words")

ex = test_examples[0]
print('\nExample question:', ex['query'])
print('Gold summary:', ex['answer'][:300])

train: 162 meetings, 1257 questions (162 general), median meeting length 6689 words
val: 35 meetings, 272 questions (35 general), median meeting length 6465 words
test: 35 meetings, 281 questions (37 general), median meeting length 6731 words

Example question: Summarize the whole meeting.
Gold summary: The meeting started with the tabling of documents, followed by petitions by various members of the house. The petitions included help for groups such as environment-protecting groups seeking assistance from the Government. The meeting then transitioned to statements by its members regarding the curr


### 2. Retrieval
Meetings are split into chunks of about 200 words, embedded with `bge-small-en-v1.5` and stored in one FAISS index per meeting (LangChain). A chunk counts as relevant if it overlaps a gold span. Only specific questions have gold spans, so these metrics use those.

In [4]:
retriever = st.Retriever()
ret = st.retrieval_metrics(test_examples, meetings_by_id, retriever, ks=(3, 5, 8, 10))
ret_df = pd.DataFrame({k: [ret[f'Hit@{k}'], ret[f'Recall@{k}'], ret[f'Precision@{k}']] for k in (3, 5, 8, 10)},
                      index=['Hit@k', 'Recall@k', 'Precision@k']).round(3)
print('MRR:', round(ret['MRR'], 3), '| questions:', ret['n_questions'])
os.makedirs('results', exist_ok=True)
ret_df.to_csv('results/retrieval.csv')
ret_df

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MRR: 0.648 | questions: 130


,3,5,8,10
Hit@k,0.738,0.838,0.915,0.931
Recall@k,0.304,0.426,0.566,0.605
Precision@k,0.410,0.351,0.307,0.268


### 3. The agent tools
The agent can call four LangChain tools when the starting chunks are not enough. Here is what each one returns for one meeting.

In [5]:
ex = next(e for e in test_examples if e['qtype'] == 'specific')
m = meetings_by_id[ex['meeting_id']]
tools = st.make_tools(m, retriever)
print('Question:', ex['query'], '\n')
print(tools['list_topics'].invoke(''), '\n')
print(tools['search_transcript'].invoke(ex['query'])[:600], '\n')
print(tools['get_topic_segment'].invoke('1')[:400])

Question: Summarize the discussion about the process of the project. 

Topics:
1. Detailed design of the remote control
2. Discussion on changing an aspect of the remote
3. Discussion on the process of the project
Speakers: Industrial Designer, Marketing, Project Manager, User Interface 

[Turns 0-0]
Project Manager: 'Kay. Hmm. Okay everybody. Welcome to the detailed design meeting. Um Let's see. Our agenda. Last time we discussed the squishy fruitiness of our remote controls and how we might pursue that. Um and I think looks like we've come up with some ideas. Um and we also talked about materials we'd use and what kind of chip would be necessary. Um so for later in the meeting I've done a a spreadsheet of production costs so we'll take a look at that. Um and see if it's changed at all from when we last discussed it. Um so actually I think Yeah um f you guys wanna give a proto 

Topic: Detailed design of the remote control
Industrial Designer: Okay well um. So our design looks somethi

### 4. Baselines
Each model is loaded once, runs its modes, then is unloaded to free GPU memory. Answers are saved line by line to `runs/`.

- **zeroshot**: first 6,000 tokens of the transcript, no retrieval
- **rag**: top-k chunks (one chunk per topic for general questions)
- **agent**: starts with the same chunks as rag, then the model decides to answer right away or to call tools to read more (ReAct loop, max 4 actions)

Each part below (4a to 4e) can run on its own: run Setup and Config, then the next cell, then the part you want. Sections 1 to 3 can be skipped. A finished part is skipped without loading the model, and an unfinished one continues from its last saved answer, so after a disconnect just run the same part again.

Rough time for 150 questions on a T4 or a similar laptop GPU: zeroshot/rag 20 to 40 min, agent 1 to 2 h per model, less for the 1.7B model.

In [6]:
if 'retriever' not in globals():
    retriever = st.Retriever()   # so the parts below work without running sections 1 to 3

def run_model(key, model_id, modes, adapter=None, stage='base'):
    # the model is loaded only if one of its runs still has questions left
    if not st.fits(key, PROFILE):
        print(f'Skipping {key}, it needs a bigger GPU than this profile allows'); return
    paths = {mode: f'runs/{key}_{stage}_{mode}.jsonl' for mode in modes}
    todo = {mode: st.count_todo(p, test_examples) for mode, p in paths.items()}
    if not any(todo.values()):
        print(f'{key}_{stage} {modes}: finished, skipped'); return
    llm = st.LLM(model_id, adapter=adapter, load_4bit=LOAD_4BIT)
    for mode, p in paths.items():
        if todo[mode]:
            st.run_experiment(llm, mode, test_examples, meetings_by_id, retriever, p, k=K, batch_size=BATCH)
    llm.unload()

def progress():
    for p in sorted(glob.glob('runs/*.jsonl')):
        left = st.count_todo(p, test_examples)
        print(f'{p}: {len(test_examples) - left} / {len(test_examples)}', '' if left else 'done')

# quick check of one answer, only before the first run (uses the example from section 3)
if not glob.glob('runs/*_base_*.jsonl') and 'm' in globals():
    llm = st.LLM(next(iter(SMALL_MODELS.values())), load_4bit=LOAD_4BIT)
    ctx, _ = st.rag_context(m, ex, retriever, k=K)
    print(llm.chat(st.summary_messages(ctx, ex['query'])))
    print('\nAgent:', st.run_agent(llm, m, ex, retriever, k=K))
    llm.unload()
progress()

runs/qwen3_4b_base_agent.jsonl: 150 / 150 done


runs/qwen3_4b_base_rag.jsonl: 150 / 150 done


runs/qwen3_4b_base_zeroshot.jsonl: 150 / 150 done


runs/qwen3_8b_base_agent.jsonl: 150 / 150 done
runs/qwen3_8b_base_rag.jsonl: 150 / 150 done


#### 4a. Small model (Qwen3-1.7B): zeroshot, rag, agent

In [7]:
for key, model_id in SMALL_MODELS.items():
    run_model(key, model_id, ['zeroshot', 'rag', 'agent'])
progress()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loaded Qwen/Qwen3-1.7B: 1.4 GB of GPU memory in use
zeroshot: 0 done, 150 to go -> runs/qwen3_17b_base_zeroshot.jsonl


  0%|          | 0/150 [00:00<?, ?it/s]

answer_zeroshot (qwen3_17b_base_zeroshot): 17.4 min, peak GPU memory 2.6 GB, 889,170 tokens, about 3,023 TFLOPs
rag: 0 done, 150 to go -> runs/qwen3_17b_base_rag.jsonl


  0%|          | 0/75 [00:00<?, ?it/s]

answer_rag (qwen3_17b_base_rag): 8.3 min, peak GPU memory 2.4 GB, 327,624 tokens, about 1,114 TFLOPs
agent: 0 done, 150 to go -> runs/qwen3_17b_base_agent.jsonl


  0%|          | 0/150 [00:00<?, ?it/s]

answer_agent (qwen3_17b_base_agent): 24.0 min, peak GPU memory 2.4 GB, 389,122 tokens, about 1,323 TFLOPs


Unloaded Qwen/Qwen3-1.7B: 0.1 GB of GPU memory still in use
runs/qwen3_17b_base_agent.jsonl: 150 / 150 done
runs/qwen3_17b_base_rag.jsonl: 150 / 150 done
runs/qwen3_17b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_4b_base_agent.jsonl: 150 / 150 done


runs/qwen3_4b_base_rag.jsonl: 150 / 150 done
runs/qwen3_4b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_8b_base_agent.jsonl: 150 / 150 done
runs/qwen3_8b_base_rag.jsonl: 150 / 150 done


#### 4b. Same-family bigger model (Qwen3-4B): zeroshot, rag, agent

In [8]:
for big in set(FAMILY.values()):
    run_model(big, BIG_MODELS[big], ['zeroshot', 'rag', 'agent'])
progress()

qwen3_4b_base ['zeroshot', 'rag', 'agent']: finished, skipped
runs/qwen3_17b_base_agent.jsonl: 150 / 150 done
runs/qwen3_17b_base_rag.jsonl: 150 / 150 done
runs/qwen3_17b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_4b_base_agent.jsonl: 150 / 150 done


runs/qwen3_4b_base_rag.jsonl: 150 / 150 done
runs/qwen3_4b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_8b_base_agent.jsonl: 150 / 150 done
runs/qwen3_8b_base_rag.jsonl: 150 / 150 done


#### 4c. Cross-family bigger model (Ministral 3 3B): rag
If Ministral fails to load, set `CROSS_FAMILY` and its `BIG_MODELS` entry in `config.py` to `meta-llama/Llama-3.2-3B-Instruct` (accept its terms on Hugging Face and log in first).

In [9]:
run_model(CROSS_FAMILY, BIG_MODELS[CROSS_FAMILY], ['rag'])
progress()

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Loaded mistralai/Ministral-3-3B-Instruct-2512-BF16: 2.5 GB of GPU memory in use (prompt ids built by the Mistral tokenizer)
rag: 0 done, 150 to go -> runs/ministral3_3b_base_rag.jsonl


  0%|          | 0/75 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=350) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=350) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=350) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (72 outputs left out)

answer_rag (ministral3_3b_base_rag): 13.5 min, peak GPU memory 3.7 GB, 344,525 tokens, about 2,343 TFLOPs


Unloaded mistralai/Ministral-3-3B-Instruct-2512-BF16: 0.1 GB of GPU memory still in use
runs/ministral3_3b_base_rag.jsonl: 150 / 150 done
runs/qwen3_17b_base_agent.jsonl: 150 / 150 done
runs/qwen3_17b_base_rag.jsonl: 150 / 150 done


runs/qwen3_17b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_4b_base_agent.jsonl: 150 / 150 done
runs/qwen3_4b_base_rag.jsonl: 150 / 150 done


runs/qwen3_4b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_8b_base_agent.jsonl: 150 / 150 done
runs/qwen3_8b_base_rag.jsonl: 150 / 150 done


#### 4d. Cross-family bigger model (Ministral 3 3B): agent

In [10]:
run_model(CROSS_FAMILY, BIG_MODELS[CROSS_FAMILY], ['agent'])
progress()

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Loaded mistralai/Ministral-3-3B-Instruct-2512-BF16: 2.5 GB of GPU memory in use (prompt ids built by the Mistral tokenizer)
agent: 0 done, 150 to go -> runs/ministral3_3b_base_agent.jsonl


  0%|          | 0/150 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=350) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=350) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=350) and `max_length`(=262144) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


... (412 outputs left out)

answer_agent (ministral3_3b_base_agent): 52.9 min, peak GPU memory 3.7 GB, 1,361,900 tokens, about 9,261 TFLOPs


Unloaded mistralai/Ministral-3-3B-Instruct-2512-BF16: 0.1 GB of GPU memory still in use
runs/ministral3_3b_base_agent.jsonl: 150 / 150 done
runs/ministral3_3b_base_rag.jsonl: 150 / 150 done
runs/qwen3_17b_base_agent.jsonl: 150 / 150 done
runs/qwen3_17b_base_rag.jsonl: 150 / 150 done


runs/qwen3_17b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_4b_base_agent.jsonl: 150 / 150 done
runs/qwen3_4b_base_rag.jsonl: 150 / 150 done
runs/qwen3_4b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_8b_base_agent.jsonl: 150 / 150 done


runs/qwen3_8b_base_rag.jsonl: 150 / 150 done


#### 4e. Reference model (Qwen3-8B): rag, agent
Needs a 16 GB GPU, so a smaller GPU skips it. Its runs were made on Colab and are reused.

In [11]:
for key in REFERENCE:
    run_model(key, BIG_MODELS[key], ['rag', 'agent'])
progress()

Skipping qwen3_8b, it needs a bigger GPU than this profile allows
runs/ministral3_3b_base_agent.jsonl: 150 / 150 done
runs/ministral3_3b_base_rag.jsonl: 150 / 150 done
runs/qwen3_17b_base_agent.jsonl: 150 / 150 done
runs/qwen3_17b_base_rag.jsonl: 150 / 150 done


runs/qwen3_17b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_4b_base_agent.jsonl: 150 / 150 done
runs/qwen3_4b_base_rag.jsonl: 150 / 150 done
runs/qwen3_4b_base_zeroshot.jsonl: 150 / 150 done
runs/qwen3_8b_base_agent.jsonl: 150 / 150 done


runs/qwen3_8b_base_rag.jsonl: 150 / 150 done


#### 4f. Timing sample for runs made before timing was added
Time per answer is a reported metric, and it is only comparable on the same GPU. Runs made elsewhere without timing are timed again here on an evenly spread sample of `TIME_SAMPLE` questions. These answers go to `results/time_samples/` and are only used for timing, not for the quality scores.

In [12]:
# zeroshot is left out: its 6,000-token prompts on the 4B model come close to the memory of a small GPU
sample = st.time_sample(test_examples)
for key, model_id in {**SMALL_MODELS, **BIG_MODELS}.items():
    old = [mode for mode in ['rag', 'agent']
           if os.path.exists(f'runs/{key}_base_{mode}.jsonl') and not st.has_timing(f'runs/{key}_base_{mode}.jsonl')]
    paths = {mode: f'results/time_samples/{key}_base_{mode}.jsonl' for mode in old}
    if not paths or not st.fits(key, PROFILE) or not any(st.count_todo(p, sample) for p in paths.values()):
        continue
    llm = st.LLM(model_id, load_4bit=LOAD_4BIT)
    for mode, p in paths.items():
        st.run_experiment(llm, mode, sample, meetings_by_id, retriever, p, k=K, batch_size=BATCH)
    llm.unload()
for p in sorted(glob.glob('results/time_samples/*.jsonl')):
    print(os.path.basename(p), f"{st.evaluate_run(p, use_bertscore=False)['sec_per_question']:.1f} s per question")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

... (7 outputs left out)

  0%|          | 0/20 [00:00<?, ?it/s]

answer_agent (qwen3_4b_base_agent): 2.6 min, peak GPU memory 3.6 GB, 48,520 tokens, about 388 TFLOPs


Unloaded Qwen/Qwen3-4B: 0.1 GB of GPU memory still in use


qwen3_4b_base_agent.jsonl 7.9 s per question
qwen3_4b_base_rag.jsonl 6.4 s per question


### 5. Quick look
ROUGE only here (fast). Full scoring with BERTScore and faithfulness is in notebook 3.

In [13]:
rows = [st.evaluate_run(p, use_bertscore=False) for p in sorted(glob.glob('runs/*_base_*.jsonl'))]
pd.DataFrame(rows).round(3)

,run,n,rouge1,rouge2,rougeL,avg_words,sec_per_question,run_minutes,gpu,tokens_per_question,gflops_per_question,tool_use_rate,avg_tool_calls,format_error_rate
0,ministral3_3b_base_agent,150,0.268,0.056,0.152,125.753,21.135,52.839,local GPU,9079.333,61739.467,0.947,1.767,0.187
1,ministral3_3b_base_rag,150,0.311,0.071,0.177,128.233,5.358,13.396,local GPU,2296.827,15618.421,NaN,NaN,NaN
2,qwen3_17b_base_agent,150,0.331,0.086,0.202,85.873,9.594,23.986,local GPU,2594.147,8820.099,0.020,0.040,0.060
3,qwen3_17b_base_rag,150,0.335,0.084,0.203,82.547,3.285,8.212,local GPU,2184.133,7426.053,NaN,NaN,NaN
4,qwen3_17b_base_zeroshot,150,0.318,0.078,0.196,92.867,6.944,17.361,local GPU,5927.800,20154.520,NaN,NaN,NaN
5,qwen3_4b_base_agent,150,0.349,0.094,0.211,83.827,NaN,NaN,NaN,NaN,NaN,0.020,0.033,0.027
6,qwen3_4b_base_rag,150,0.338,0.097,0.205,108.773,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,qwen3_4b_base_zeroshot,150,0.322,0.085,0.195,115.880,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,qwen3_8b_base_agent,150,0.353,0.100,0.214,91.907,NaN,NaN,NaN,NaN,NaN,0.460,1.207,0.333
9,qwen3_8b_base_rag,150,0.353,0.099,0.215,94.047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
# one agent run that used tools, to see which ones it picked
agent_rows = st.read_jsonl(sorted(glob.glob('runs/*_base_agent.jsonl'))[0])
r = next((x for x in agent_rows if x['trace']), agent_rows[0])
print('Question:', r['query'])
for step in r['trace']:
    print('  ->', step['tool'], '|', step['input'])
print('\nAnswer:', r['pred'])
print('\nGold:', r['answer'])

Question: Summarize the whole meeting.
  -> list_topics | 
  -> get_topic_segment | Presenting petitions
  -> get_topic_segment | Statements by Members
  -> get_topic_segment | Questions to Ministers

Answer: This meeting appears to be a **parliamentary session** where key segments included:
1. **Petitions** were presented, with reminders about certification and conciseness from the Chair.
2. **Statements by members** highlighted COVID-19 resilience, local innovations, and social justice issues. Notable speakers included **Paul Lefebvre** (Lib.), who praised Sudbury’s COVID-19 efforts and local innovations like ventilators and hand sanitizer, and **Ms. Leah Gazan** (NDP), who advocated for Black Lives Winnipeg and systemic change against police brutality and racism.
3. **Ministerial questions** focused on accountability and policy responses. For example:
   - **Mr. Pat Kelly** (CPC) questioned **Steven Guilbeault** (Heritage) about the Auditor General’s budget cuts and funding needs, w